# 🚀 Colab GGUF Model Runner
Run open-source GGUF language models inside Google Colab with GPU acceleration and an OpenAI-compatible FastAPI server.

In [ ]:
# @title 1. Environment & GPU Setup
import sys
import subprocess
import os

print("🔍 Checking GPU availability...")
gpu_status = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if gpu_status.returncode != 0:
    print("⚠️ WARNING: No NVIDIA GPU detected! Please go to Runtime -> Change runtime type -> Select T4/V100/A100 GPU.")
else:
    print("✅ GPU detected successfully:")
    for line in gpu_status.stdout.splitlines()[:8]:
        print("   " + line)

print("\n📦 Installing dependencies (optimized llama-cpp-python with CUDA support)...")
env = os.environ.copy()
env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "nest_asyncio", "pyngrok", "requests", "pydantic", "tqdm"])
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "llama-cpp-python", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121", "--no-cache-dir"])
except Exception:
    print("Falling back to compiling llama-cpp-python from source with CUDA...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "llama-cpp-python", "--no-cache-dir"], env=env)
print("✅ All dependencies installed successfully!")

In [ ]:
# @title 2. Data Input & Configuration Form
# @markdown Specify the model download link and runtime options.
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/main/qwen2.5-coder-7b-instruct-q4_k_m.gguf"  # @param {type:"string"}
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER_NAME = "colab_models"  # @param {type:"string"}
NGROK_AUTHTOKEN = ""  # @param {type:"string"}
N_GPU_LAYERS = -1  # @param {type:"integer"}
CONTEXT_SIZE = 4096  # @param {type:"integer"}

import os
from pathlib import Path
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        TARGET_DIR = Path('/content/drive/MyDrive') / DRIVE_FOLDER_NAME
        TARGET_DIR.mkdir(parents=True, exist_ok=True)
        print(f"📁 Storage set to Google Drive: {TARGET_DIR}")
    except Exception as e:
        print(f"⚠️ Failed to mount Google Drive ({e}). Falling back to local storage.")
        TARGET_DIR = Path('/content/models')
        TARGET_DIR.mkdir(parents=True, exist_ok=True)
else:
    TARGET_DIR = Path('/content/models')
    TARGET_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Storage set to local environment: {TARGET_DIR}")
raw_filename = MODEL_URL.split('?')[0].split('/')[-1]
if not raw_filename.endswith('.gguf'):
    raw_filename = 'model.gguf'
LOCAL_MODEL_PATH = TARGET_DIR / raw_filename
print(f"Target model file: {LOCAL_MODEL_PATH}")

In [ ]:
# @title 3. Download Model & Verify Integrity
import os
import requests
from tqdm.auto import tqdm
def download_file_with_progress(url: str, destination: Path):
    if destination.exists() and destination.stat().st_size > 10 * 1024 * 1024:
        print(f"⚡ Model already exists at: {destination} ({destination.stat().st_size / (1024**3):.2f} GB). Skipping download.")
        return True
    print(f"📥 Downloading model from: {url}")
    headers = {"User-Agent": "ColabModelRunner/1.0"}
    with requests.get(url, stream=True, headers=headers, allow_redirects=True) as response:
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        with open(destination, 'wb') as file, tqdm(desc=destination.name, total=total_size, unit='iB', unit_scale=True, unit_divisor=1024) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    bar.update(len(chunk))
    file_size_gb = destination.stat().st_size / (1024 ** 3)
    print(f"\n✅ Download completed: {destination.name} ({file_size_gb:.2f} GB)")
    return True
try:
    download_file_with_progress(MODEL_URL, LOCAL_MODEL_PATH)
    assert LOCAL_MODEL_PATH.exists() and LOCAL_MODEL_PATH.stat().st_size > 1024 * 1024, "Model file is missing or corrupted!"
    print("✅ Model integrity verified.")
except Exception as err:
    print(f"❌ Download failed: {err}")
    raise err

In [ ]:
# @title 4. FastAPI Server & OpenAI-Compatible Endpoint with Tunneling
import secrets
import time
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Depends, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
from llama_cpp import Llama
nest_asyncio.apply()
API_KEY = f"sk-colab-{secrets.token_hex(16)}"
security_scheme = HTTPBearer()
def verify_api_key(credentials: HTTPAuthorizationCredentials = Security(security_scheme)):
    if credentials.scheme.lower() != 'bearer' or credentials.credentials != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing Bearer API Key")
    return credentials.credentials
llm = Llama(model_path=str(LOCAL_MODEL_PATH), n_gpu_layers=N_GPU_LAYERS, n_ctx=CONTEXT_SIZE, verbose=False)
app = FastAPI(title="Colab Model Runner", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = 'default'
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    max_tokens: Optional[int] = 512
    stream: Optional[bool] = False
@app.get("/")
def root():
    return {"status": "online", "model": LOCAL_MODEL_PATH.name}

@app.get("/v1/models", dependencies=[Depends(verify_api_key)])
def list_models():
    return {"object": "list", "data": [{"id": LOCAL_MODEL_PATH.stem, "object": "model", "owned_by": "colab-runner"}]}
@app.post("/v1/chat/completions", dependencies=[Depends(verify_api_key)])
def chat_completions(req: ChatCompletionRequest):
    try:
        formatted_messages = [{"role": m.role, "content": m.content} for m in req.messages]
        return llm.create_chat_completion(messages=formatted_messages, temperature=req.temperature, top_p=req.top_p, max_tokens=req.max_tokens)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
public_url = None
if NGROK_AUTHTOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    public_url = ngrok.connect(8000).public_url
else:
    import subprocess
    subprocess.Popen(["npx", "localtunnel", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    public_url = "http://localhost:8000 (Set NGROK_AUTHTOKEN for external public link)"
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
base_endpoint = public_url.rstrip('/') if public_url.startswith('http') else 'http://localhost:8000'
print(f"🌐 Base URL   : {base_endpoint}")
print(f"🔑 API Key    : {API_KEY}")
print(f"⚡ Endpoint   : {base_endpoint}/v1/chat/completions")
curl_example = f'''curl -X POST "{base_endpoint}/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer {API_KEY}" \
  -d '{{
    "messages": [{"role": "user", "content": "Hello!"}],
    "max_tokens": 128
  }}' '''
print(curl_example)
print("Keep this Colab tab open to maintain the active server.")